# Convolutional Neural Network

In [14]:
import torch, torchvision, re, os
import numpy as np
from PIL import Image

DATA_DIR = '../data/images'

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
])

images, climb_ids, angles, labels = [], [], [], []
for filename in sorted(os.listdir(DATA_DIR)):
    # Images are titled 'id=[id]_angle=[angle]_grade=[grade].png'
    id, angle, grade = re.findall(r'id=(.+)_angle=(.+)_grade=(.+)\.png', filename)[0]
    climb_ids.append(id)
    angles.append(int(angle))
    labels.append(int(grade) - 10) # Standardize to [0, 22]
    
    img = Image.open(os.path.join(DATA_DIR, filename)).convert('RGB')
    images.append(preprocess(img).numpy())
images =  np.stack(images)
grades = np.array(labels, dtype=np.int64)

# Separate train/val/test splits
images_train = images[:80]; grades_train = grades[:80]
images_val = images[80:]; grades_val = grades[80:]

# # Gather climb grades
# climbs_conn = sqlite3.connect('../data/climbs.sqlite3')
# climbs_cursor = climbs_conn.cursor()

# # Get the list of climbs we are working with (>= 5 ascents, Kilter Board Original (layout 10))
# climbs_cursor.execute("""SELECT currentDifficultyId FROM climb_stats
#                                   INNER JOIN climbs
#                                   ON climb_stats.climbUuid = climbs.climbUuid
#                                   WHERE ascentCount >= 5
#                                   AND climb_stats.product_layout_uuid = '10'
#                                   AND currentDifficultyId >= 10;""")

# # Extract grade data
# rows = climbs_cursor.fetchall()
# climb_grades = np.array(list(map(lambda x: int(x[0]) - 10, rows))) # Standardize to [0, 22]

# train_grades = climb_grades[:1000] # truncate to 1000 entries for now
# val_grades = climb_grades[1000:1201]

# # Images are each 216 x 234 pixels with 3 RGB channels
# images_train = np.array([[[[0.5] * 234] * 216] * 3] * len(train_grades), dtype=np.float32)
# images_val = np.array([[[[0.5] * 234] * 216] * 3] * len(val_grades), dtype=np.float32)
# print(images_train.shape)

# TODO: preprocess images in some way?

In [15]:
# (for now) simple 2 layer Convolution with average pooling and 1 fully connected layer
class CNN(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = torch.nn.Conv2d(3, 64, 3, padding='same')
        self.pool1 = torch.nn.AvgPool2d(2, 2)
        self.conv2 = torch.nn.Conv2d(64, 128, 3, padding='same')
        self.pool2 = torch.nn.AvgPool2d(2, 2)
        self.fc = torch.nn.Linear(400896, 23) # 128 * 54 * 58 input features
        self.flatten = torch.nn.Flatten()
        self.relu = torch.nn.ReLU()
    
    def forward(self, X):
        c1 = self.conv1(X)
        r1 = self.relu(c1)
        p1 = self.pool1(r1)
        
        c2 = self.conv2(p1)
        r2 = self.relu(c2)
        p2 = self.pool2(r2)
        
        f = self.flatten(p2)
        y = self.fc(f)
        return y

In [8]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [25]:
from torch.utils.data import TensorDataset, DataLoader

# Hyperparameters
batch_size = 64
epsilon = 1E-6

model = CNN().to(device)

# Process dataset
train_dataset = TensorDataset(torch.from_numpy(images_train), torch.from_numpy(grades_train))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(torch.from_numpy(images_val), torch.from_numpy(grades_val)), batch_size=batch_size)
# test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(Y_test)), batch_size=batch_size)

# Use CCE loss
criterion = torch.nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

In [10]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 9296279


In [26]:
import plotly.graph_objects as go
import time

# Taken from Lesson 19
def train_model(model, optimizer, num_epochs, checkpoint_path):
    """Train, keep the checkpoint with the lowest validation loss, and plot the losses."""
    training_losses = []
    validation_losses = []
    best_validation_loss = float('inf')
    best_validation_epoch = -1
    training_start = time.time()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        num_batches = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            optimizer.zero_grad() 
            loss = criterion(outputs, labels)
            loss.backward() 
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        training_losses.append(avg_loss)

        # Compute validation loss. Don't need gradients here.
        model.eval()
        with torch.no_grad():
            val_total = 0
            for images, labels in val_loader:
                val_outputs = model(images.to(device))
                val_total += criterion(val_outputs, labels.to(device)).item() * len(labels)
            validation_loss = val_total / len(images_val)
            validation_losses.append(validation_loss)

        # Save the model if this is the best validation loss so far
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_validation_epoch = epoch
            torch.save(model.state_dict(), checkpoint_path)
            best_marker = ' *'
        else:
            best_marker = ''

        if epoch % (num_epochs / 10) == 0:
            print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {avg_loss:.5f}, Val Loss: {validation_loss:.5f}{best_marker}')

    total_time = time.time() - training_start
    print(f'Average time per epoch: {total_time / num_epochs:.2f}s')
    print(f'Best validation loss: {best_validation_loss:.5f} (epoch {best_validation_epoch})')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=list(range(len(training_losses))), y=training_losses, mode='lines', name='Training Loss'))
    fig.add_trace(go.Scatter(x=list(range(len(validation_losses))), y=validation_losses, mode='lines', name='Validation Loss'))
    fig.update_layout(title="Training and Validation Loss Over Time", xaxis_title="Epoch", yaxis_title="Loss")
    fig.show()


train_model(model, optimizer, num_epochs=100, checkpoint_path='kilter_cnn.pth')

Epoch [0/100], Train Loss: 3.15527, Val Loss: 3.14108 *
Epoch [10/100], Train Loss: 2.94929, Val Loss: 3.00325 *
Epoch [20/100], Train Loss: 2.78084, Val Loss: 2.89624 *
Epoch [30/100], Train Loss: 2.65765, Val Loss: 2.80636 *
Epoch [40/100], Train Loss: 2.50422, Val Loss: 2.73153 *
Epoch [50/100], Train Loss: 2.48310, Val Loss: 2.68482 *
Epoch [60/100], Train Loss: 2.42336, Val Loss: 2.65015 *
Epoch [70/100], Train Loss: 2.39025, Val Loss: 2.61835 *
Epoch [80/100], Train Loss: 2.38805, Val Loss: 2.60268 *
Epoch [90/100], Train Loss: 2.29552, Val Loss: 2.60243
Average time per epoch: 0.50s
Best validation loss: 2.59959 (epoch 85)


In [7]:
climbs_cursor.close()